<a href="https://colab.research.google.com/github/leespace1231/leespace1231.github.io/blob/main/%EA%B3%BC%EC%A0%9C7_4_22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nest_asyncio

In [ ]:
pip install fastapi uvicorn python-multipart httpx beautifulsoup4 gradio

In [ ]:
import sqlite3
import httpx
import nest_asyncio
import gradio as gr
import pandas as pd
from bs4 import BeautifulSoup

nest_asyncio.apply()

def init_db():
    conn = sqlite3.connect("quotes.db", check_same_thread=False)
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS quotes")
    cursor.execute('''CREATE TABLE quotes
                      (id INTEGER PRIMARY KEY AUTOINCREMENT,
                       text TEXT, author TEXT, category TEXT)''')
    conn.commit()
    conn.close()

# --- [수정된 수집 로직: 2페이지까지 크롤링] ---
async def crawl_quotes(category="love"):
    if not category: return "❌ 카테고리를 입력하세요."

    collected_quotes = []
    page = 1

    try:
        async with httpx.AsyncClient() as client:
            # 20개를 채울 때까지 최대 2페이지까지 탐색
            while len(collected_quotes) < 20 and page <= 2:
                url = f"https://quotes.toscrape.com/tag/{category}/page/{page}/"
                response = await client.get(url, timeout=10.0)
                soup = BeautifulSoup(response.text, 'html.parser')
                quotes_elements = soup.find_all('div', class_='quote')

                if not quotes_elements: # 더 이상 페이지가 없으면 중단
                    break

                for q in quotes_elements:
                    if len(collected_quotes) >= 20: break
                    text = q.find('span', class_='text').text
                    author = q.find('small', class_='author').text
                    collected_quotes.append((text, author, category))

                page += 1

        # DB 저장
        conn = sqlite3.connect("quotes.db", check_same_thread=False)
        cursor = conn.cursor()
        cursor.execute("DELETE FROM quotes")
        cursor.execute("DELETE FROM sqlite_sequence WHERE name='quotes'")

        for quote in collected_quotes:
            cursor.execute("INSERT INTO quotes (text, author, category) VALUES (?, ?, ?)", quote)

        conn.commit()
        conn.close()
        return f"✅ '{category}' 카테고리에서 총 {len(collected_quotes)}개 수집 완료!"
    except Exception as e:
        return f"❌ 오류: {str(e)}"

def get_data():
    try:
        conn = sqlite3.connect("quotes.db", check_same_thread=False)
        # 1번부터 20번까지 순차적으로 출력
        query = "SELECT id, text, author, category FROM quotes ORDER BY id ASC LIMIT 20"
        df = pd.read_sql_query(query, conn)
        conn.close()

        if not df.empty:
            # 긴 텍스트 요약
            df['text'] = df['text'].apply(lambda x: x[:60] + "..." if len(x) > 60 else x)
            return df
        return pd.DataFrame(columns=["id", "text", "author", "category"])
    except:
        return "데이터가 없습니다."

# Gradio UI (기존과 동일)
with gr.Blocks() as demo:
    gr.Markdown("# 💡 격언 관리 시스템")
    with gr.Tab("📊 데이터 조회"):
        output_df = gr.Dataframe(headers=["id", "격언 내용", "저자", "카테고리"])
        refresh_btn = gr.Button("🔄 20개 데이터 확인")
        refresh_btn.click(fn=get_data, outputs=output_df)
    with gr.Tab("🚀 데이터 수집"):
        cat_input = gr.Textbox(label="카테고리", value="love")
        crawl_btn = gr.Button("수집 시작 (2페이지까지 탐색)")
        status_out = gr.Textbox(label="상태")
        crawl_btn.click(fn=crawl_quotes, inputs=cat_input, outputs=status_out)

if __name__ == "__main__":
    init_db()
    demo.queue().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6d42a343ae72bf227.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


본 시스템은 다음과 같은 3단계 프로세스로 작동합니다.

1. 수집: BeautifulSoup을 이용해 외부 격언 사이트의 여러 페이지를 탐색, 20개의 데이터를 추출합니다.

2. 저장: SQLite3를 활용해 테이블을 설계하고, 수집 시마다 인덱스를 초기화하여 데이터의 정결성을 유지합니다.

3. 제공: FastAPI에 마운트된 Gradio UI를 통해 사용자가 외부 URL로 접속하여 실시간 요약된 데이터를 조회할 수 있도록 구현했습니다."